In [1]:
!uname -a && cat /etc/*release
!pwd
!ls -la

Linux a8e8c0c18140 6.6.113+ #1 SMP Mon Feb  2 12:27:57 UTC 2026 x86_64 x86_64 x86_64 GNU/Linux
DISTRIB_ID=Ubuntu
DISTRIB_RELEASE=22.04
DISTRIB_CODENAME=jammy
DISTRIB_DESCRIPTION="Ubuntu 22.04.5 LTS"
PRETTY_NAME="Ubuntu 22.04.5 LTS"
NAME="Ubuntu"
VERSION_ID="22.04"
VERSION="22.04.5 LTS (Jammy Jellyfish)"
VERSION_CODENAME=jammy
ID=ubuntu
ID_LIKE=debian
HOME_URL="https://www.ubuntu.com/"
SUPPORT_URL="https://help.ubuntu.com/"
BUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"
PRIVACY_POLICY_URL="https://www.ubuntu.com/legal/terms-and-policies/privacy-policy"
UBUNTU_CODENAME=jammy
/content
total 16
drwxr-xr-x 1 root root 4096 Feb  6 14:31 .
drwxr-xr-x 1 root root 4096 Mar 19 10:35 ..
drwxr-xr-x 4 root root 4096 Feb  6 14:31 .config
drwxr-xr-x 1 root root 4096 Feb  6 14:31 sample_data


In [2]:
cell_str='''
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <cuda_runtime.h>
#include <cublas_v2.h> // Ecco la libreria magica!

int main(int argc, char **argv) {
    if (argc < 2) {
        fprintf(stderr, "Uso: %s <n>\\n", argv[0]);
        return 1;
    }

    int n = atoi(argv[1]);
    size_t bytes = n * n * sizeof(float);

    float *h_a = (float*)malloc(bytes);
    float *h_b = (float*)malloc(bytes);
    float *h_c = (float*)malloc(bytes);

    for (int i = 0; i < n; i++) {
        for (int j = 0; j < n; j++) {
            h_a[i * n + j] = 2.0f;
            h_b[i * n + j] = 3.0f;
            h_c[i * n + j] = 0.0f;
        }
    }

    float *d_a, *d_b, *d_c;
    cudaMalloc(&d_a, bytes);
    cudaMalloc(&d_b, bytes);
    cudaMalloc(&d_c, bytes);

    cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice);

    // --- INIZIO MAGIA CUBLAS ---

    // 1. Creiamo un "handle", ovvero il contesto per la libreria
    cublasHandle_t handle;
    cublasCreate(&handle);

    // cuBLAS calcola un'operazione generica: C = (alpha * A * B) + (beta * C)
    // Noi vogliamo solo C = A * B, quindi impostiamo alpha a 1 e beta a 0
    float alpha = 1.0f;
    float beta = 0.0f;

    clock_t start = clock();

    // 2. Lanciamo la funzione Sgemm (Single precision GEneral Matrix Multiply)
    // TRUCCHETTO: cuBLAS usa il formato "Column-Major" (colonne in memoria, stile Fortran).
    // Noi in C usiamo "Row-Major" (righe in memoria). Per fare in modo che la matematica
    // torni perfetta senza dover riordinare a mano le matrici, basta passare a cuBLAS
    // prima la matrice B e poi la matrice A!
    cublasSgemm(handle, CUBLAS_OP_N, CUBLAS_OP_N,
                n, n, n,
                &alpha,
                d_b, n,  // <--- Nota: passiamo d_b per primo!
                d_a, n,
                &beta,
                d_c, n);

    cudaDeviceSynchronize();
    clock_t end = clock();

    // 3. Chiudiamo l'handle
    cublasDestroy(handle);

    // --- FINE MAGIA CUBLAS ---

    printf("Execution Time (cuBLAS FP32): %.4f seconds\\n", (double)(end - start) / CLOCKS_PER_SEC);

    cudaMemcpy(h_c, d_c, bytes, cudaMemcpyDeviceToHost);

    // Stampa di controllo
    FILE *f = fopen("mat-res.txt", "w");
    if (f) {
        fprintf(f, "%d\\n\\n", n);
        int sample = (n < 100) ? n : 10;
        for (int i = 0; i < sample; i++) {
            for (int j = 0; j < sample; j++) {
                fprintf(f, "%.0f ", h_c[i * n + j]);
            }
            fprintf(f, "\\n");
        }
        fclose(f);
    }

    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    free(h_a); free(h_b); free(h_c);

    return 0;
}
'''
with open('cuda_matrixmult_cuBLAS.cu', 'w') as f:
    f.write(cell_str)

In [3]:
!nvcc -arch=sm_75 -O3 -lcublas cuda_matrixmult_cuBLAS.cu -o cuda_matrixmult_cuBLAS

In [4]:
!nvprof ./cuda_matrixmult_cuBLAS 5000

==5313== NVPROF is profiling process 5313, command: ./cuda_matrixmult_cuBLAS 15000
Execution Time (cuBLAS FP32): 1.3492 seconds
==5313== Profiling application: ./cuda_matrixmult_cuBLAS 15000
==5313== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   67.93%  1.35280s         1  1.35280s  1.35280s  1.35280s  volta_sgemm_128x128_nn
                   21.54%  428.90ms         2  214.45ms  210.63ms  218.28ms  [CUDA memcpy HtoD]
                   10.53%  209.66ms         1  209.66ms  209.66ms  209.66ms  [CUDA memcpy DtoH]
      API calls:   58.50%  1.35282s         5  270.56ms  1.1860us  1.35280s  cudaDeviceSynchronize
                   27.70%  640.50ms         3  213.50ms  209.99ms  218.51ms  cudaMemcpy
                   10.07%  232.77ms         6  38.796ms  16.382us  232.02ms  cudaMalloc
                    2.07%  47.961ms         1  47.961ms  47.961ms  47.961ms  cudaGetSymbolAddress
                    0.65%  15.102ms

In [5]:
!nvprof ./cuda_matrixmult_cuBLAS 10000

==5481== NVPROF is profiling process 5481, command: ./cuda_matrixmult_cuBLAS 10000
Execution Time (cuBLAS FP32): 0.4080 seconds
==5481== Profiling application: ./cuda_matrixmult_cuBLAS 10000
==5481== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   59.97%  399.54ms         1  399.54ms  399.54ms  399.54ms  volta_sgemm_128x64_nn
                   26.00%  173.26ms         2  86.629ms  86.607ms  86.651ms  [CUDA memcpy HtoD]
                   14.03%  93.476ms         1  93.476ms  93.476ms  93.476ms  [CUDA memcpy DtoH]
      API calls:   45.21%  399.57ms         5  79.913ms  1.1140us  399.55ms  cudaDeviceSynchronize
                   30.28%  267.63ms         3  89.210ms  86.869ms  93.875ms  cudaMemcpy
                   20.56%  181.70ms         6  30.283ms  23.479us  181.24ms  cudaMalloc
                    1.47%  12.947ms         1  12.947ms  12.947ms  12.947ms  cudaGetSymbolAddress
                    0.91%  8.0062ms 

In [6]:
!nvprof ./cuda_matrixmult_cuBLAS 15000

==5501== NVPROF is profiling process 5501, command: ./cuda_matrixmult_cuBLAS 15000
Execution Time (cuBLAS FP32): 1.2998 seconds
==5501== Profiling application: ./cuda_matrixmult_cuBLAS 15000
==5501== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   67.25%  1.31273s         1  1.31273s  1.31273s  1.31273s  volta_sgemm_128x128_nn
                   22.10%  431.36ms         2  215.68ms  214.53ms  216.83ms  [CUDA memcpy HtoD]
                   10.65%  207.97ms         1  207.97ms  207.97ms  207.97ms  [CUDA memcpy DtoH]
      API calls:   59.97%  1.31278s         5  262.56ms  1.3380us  1.31273s  cudaDeviceSynchronize
                   29.25%  640.17ms         3  213.39ms  208.32ms  217.06ms  cudaMemcpy
                    8.65%  189.44ms         6  31.574ms  7.1000us  188.79ms  cudaMalloc
                    0.84%  18.318ms         1  18.318ms  18.318ms  18.318ms  cudaGetSymbolAddress
                    0.50%  10.901ms

In [7]:
!nvprof ./cuda_matrixmult_cuBLAS 20000

==5529== NVPROF is profiling process 5529, command: ./cuda_matrixmult_cuBLAS 20000
Execution Time (cuBLAS FP32): 3.1148 seconds
==5529== Profiling application: ./cuda_matrixmult_cuBLAS 20000
==5529== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   73.73%  3.10768s         1  3.10768s  3.10768s  3.10768s  volta_sgemm_128x128_nn
                   17.62%  742.79ms         2  371.39ms  341.16ms  401.62ms  [CUDA memcpy HtoD]
                    8.65%  364.44ms         1  364.44ms  364.44ms  364.44ms  [CUDA memcpy DtoH]
      API calls:   69.81%  3.10771s         5  621.54ms  1.7790us  3.10769s  cudaDeviceSynchronize
                   24.89%  1.10815s         3  369.38ms  341.37ms  401.85ms  cudaMemcpy
                    4.40%  195.77ms         6  32.628ms  3.9310us  195.25ms  cudaMalloc
                    0.35%  15.736ms         1  15.736ms  15.736ms  15.736ms  cudaGetSymbolAddress
                    0.21%  9.5379ms